# 02 ContinueVariable EDA

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy import stats

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"

BEHAVIOR_VAR = "EverCigaretteUse"
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"

processed = pd.read_csv(PROCESSED_PATH)
print("Processed data shape:", processed.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\asd45\\Downloads\\combine\\data\\processed\\yrbs_selected_cleaned.csv'

In [ ]:
#copy
behavior = processed[[BEHAVIOR_VAR, "behavior_binary"]].copy()
behavior_valid = behavior.dropna(subset=["behavior_binary"]).copy()

cont = processed[[CONT_VAR]].copy()
cont_valid = cont.dropna(subset=[CONT_VAR]).copy()

## 2.Continuous Variable EDA

### 2.1 Statistic and missing or invalid value count

In [ ]:
w = cont_valid[CONT_VAR]
summary_stats = pd.DataFrame({
    "statistic": ["valid size", "mean", "median", "sd", "min", "Q1", "Q3", "max", "Missing  or invalid"],
    "value": [
        int(w.shape[0]),
        float(w.mean()),
        float(w.median()),
        float(w.std(ddof=1)),
        float(w.min()),
        float(w.quantile(0.25)),
        float(w.quantile(0.75)),
        float(w.max()),
        int(processed[CONT_VAR].isna().sum())        
    ]
})
summary_stats.to_csv(TAB_DIR / "02_continuous_summary_stats.csv", index=False)
display(summary_stats)

**Observation**: The sample mean is 68.55 kg and the median is 65.32 kg. Because the mean is above the median, the distribution appears somewhat right-skewed.

### 2.2 Histogram

In [ ]:
plt.figure(figsize=(8, 4.8))
sns.histplot(w, bins=30, kde=True, alpha=0.75)

plt.axvline(w.mean(), linestyle="--", linewidth=2, label=f"Mean = {w.mean():.2f}")
plt.axvline(w.median(), linestyle=":", linewidth=2, label=f"Median = {w.median():.2f}")

plt.title("Weight Distribution", fontsize=14, pad=10)
plt.xlabel("Weight (kg)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()

plt.savefig(FIG_DIR / "02_weight_histogram.png", bbox_inches="tight")
plt.show()

**Observation:** The histogram shows a single main cluster with a longer right tail. This supports the idea that unusually high weights pull the mean upward.

### 2.3 Boxplot

In [ ]:
q1 = float(w.quantile(0.25))
q3 = float(w.quantile(0.75))
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = int(((w < lower) | (w > upper)).sum())

outlier_table = pd.DataFrame({
    "metric": ["Q1", "Q3", "IQR", "Lower fence", "Upper fence", "Outlier count"],
    "value": [q1, q3, iqr, lower, upper, outliers]
})
outlier_table.to_csv(TAB_DIR / "02_weight_outlier_check.csv", index=False)
display(outlier_table)

sns.set_theme(style="whitegrid", context="notebook")

plt.figure(figsize=(8, 3.2))
sns.boxplot(x=w, width=0.35)

plt.axvline(w.mean(), linestyle="--", linewidth=2, label=f"Mean = {w.mean():.2f}")
plt.axvline(w.median(), linestyle=":", linewidth=2, label=f"Median = {w.median():.2f}")

plt.title("Weight Boxplot", fontsize=13, pad=10)
plt.xlabel("Weight (kg)", fontsize=11)
plt.legend(frameon=True, fontsize=9)
plt.tight_layout()

plt.savefig(FIG_DIR / "02_weight_boxplot.png", bbox_inches="tight")
plt.show()

**Observation:** The IQR rule flags 447 possible outliers. So the weight variable is not perfectly symmetric, but the sample size is very large, which helps make one-sample t procedures reasonably stable.

## Test: Simulate whether T-Test is true or not

In [ ]:
# ===== 0. 讀取你目前已清理好的體重資料 =====
w = processed[CONT_VAR].dropna().to_numpy(dtype=float)
mu0 = 68.0
alpha = 0.05
B = 3000
sample_sizes = [20, 50, 100, 10000]

print("=== Raw weight data check ===")
print(f"n = {len(w)}")
print(f"mean = {w.mean():.4f}")
print(f"median = {np.median(w):.4f}")
print(f"sd = {w.std(ddof=1):.4f}")
print(f"skewness = {stats.skew(w, bias=False):.4f}")

# ===== 1. 原始分布：Histogram + QQ plot =====
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Histogram
sns.histplot(w, bins=30, kde=True, alpha=0.75, ax=axes[0])
axes[0].axvline(w.mean(), linestyle="--", linewidth=2, label=f"Mean = {w.mean():.2f}")
axes[0].axvline(np.median(w), linestyle=":", linewidth=2, label=f"Median = {np.median(w):.2f}")
axes[0].set_title("Raw Weight Distribution", fontsize=13, pad=10)
axes[0].set_xlabel("Weight (kg)", fontsize=11)
axes[0].set_ylabel("Frequency", fontsize=11)
axes[0].legend(frameon=True, fontsize=10)
axes[0].grid(alpha=0.25)

# QQ plot（自己畫，比 probplot 的預設好看）
theoretical_q, ordered_vals = stats.probplot(w, dist="norm", fit=False)
slope, intercept, _ = stats.probplot(w, dist="norm")[1]

axes[1].scatter(theoretical_q, ordered_vals, s=18, alpha=0.8)
xline = np.linspace(min(theoretical_q), max(theoretical_q), 200)
axes[1].plot(xline, slope * xline + intercept, linewidth=2)
axes[1].set_title("QQ Plot of Raw Weight", fontsize=13, pad=10)
axes[1].set_xlabel("Theoretical quantiles", fontsize=11)
axes[1].set_ylabel("Ordered values", fontsize=11)
axes[1].grid(alpha=0.25)

sns.despine()
plt.tight_layout()
plt.show()

# ===== 2. 建立「保留原始形狀，但真實平均數等於 mu0」的資料 =====
w_null = w - w.mean() + mu0

print("\n=== Null-centered data check ===")
print(f"shifted mean = {w_null.mean():.4f} (should be close to {mu0:.1f})")
print(f"shifted skewness = {stats.skew(w_null, bias=False):.4f}")

# ===== 3. 模擬不同樣本數下，樣本平均分布 & t-test 拒絕率 =====
rng = np.random.default_rng(13)

results = []
sample_means_dict = {}

for n in sample_sizes:
    means = np.empty(B)
    pvals = np.empty(B)

    for b in range(B):
        sample = rng.choice(w_null, size=n, replace=True)
        means[b] = sample.mean()
        _, pvals[b] = stats.ttest_1samp(sample, popmean=mu0)

    sample_means_dict[n] = means

    results.append({
        "n": n,
        "mean_of_sample_means": means.mean(),
        "sd_of_sample_means": means.std(ddof=1),
        "skewness_of_sample_means": stats.skew(means, bias=False),
        "rejection_rate_at_5pct": (pvals < alpha).mean()
    })

results_df = pd.DataFrame(results)
print("\n=== Simulation summary ===")
display(results_df.round(4))

# ===== 4. 不同樣本數下，樣本平均分布 =====
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
axes = axes.ravel()

for ax, n in zip(axes, sample_sizes):
    means = sample_means_dict[n]
    sns.histplot(means, bins=28, kde=True, alpha=0.75, ax=ax)
    ax.axvline(mu0, linestyle="--", linewidth=2, label=f"mu0 = {mu0:.1f}")
    ax.set_title(
        f"Sampling Distribution of Mean (n = {n})\nSkewness = {stats.skew(means, bias=False):.3f}",
        fontsize=11,
        pad=8
    )
    ax.set_xlabel("Sample mean", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    ax.legend(frameon=True, fontsize=9)
    ax.grid(alpha=0.25)

sns.despine()
plt.tight_layout()
plt.show()

# ===== 5. 簡短文字解讀 =====
print("\n=== How to read the results ===")
print("1. 如果 raw data 的 skewness > 0，代表原始體重分布有右偏傾向。")
print("2. 如果 sample mean distribution 的 skewness 隨 n 變大而變小，代表樣本平均越來越接近常態。")
print("3. 如果 rejection_rate_at_5pct 在大樣本時接近 0.05，代表 one-sample t-test 在這種偏態下仍然算穩健。")